In [1]:
pip install google-play-scraper pandas 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 326.5 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 865.7 kB/s eta 0:00:00a 0:00:01
  Using cached numpy-2.4.4-cp312-cp312-macosx_10_13_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 852.6 kB/s eta 0:00:0000:0100:01
Using cached numpy-2.4.4-cp312-cp312-macosx_10_13_x86_64.whl (16.7 MB)

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


BOA

In [ ]:
# The unique identifier for BOA Bank's app on the Google Play Store
BOA_APP_ID = 'com.boa.boaMobileBanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info1 = app(
    BOA_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("BOA Bank App Info")
print("=" * 50)
print(f"App Title   : {app_info1['title']}")
print(f"Current Score: {app_info1['score']}")
print(f"Total Ratings: {app_info1['ratings']:,}")
print(f"Total Reviews: {app_info1['reviews']:,}")
print(f"Installs     : {app_info1['installs']}")



BOA Bank App Info
App Title   : BoA Mobile
Current Score: 4.3978496
Total Ratings: 9,276
Total Reviews: 1,465
Installs     : 1,000,000+


scraping reviews

In [3]:
# Step 2: Scrape reviews
print(f"Scraping reviews for BOA Bank...")

result1, continuation_token1 = reviews(
    BOA_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=500,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result1)} raw reviews")

Scraping reviews for BOA Bank...
Collected 500 raw reviews


In [4]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result1[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result1[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 9dd3879e-2b0c-4058-8c8c-19f6b376f69c
  userName: Abel Legesse
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjVlAiPqwEucytjQHmhCbqlROOv44ryrgg5M6NZ2fb1qyojQN2u1
  content: The worst app, also bank am begging for my own money
  score: 1
  thumbsUpCount: 0
  reviewCreatedVersion: 26.05.11
  at: 2026-05-16 12:35:22
  replyContent: None
  repliedAt: None
  appVersion: 26.05.11


In [5]:
# Step 3: Extract only the columns we need
raw_data1 = []

for r in result1:
    raw_data1.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'BOA Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw1 = pd.DataFrame(raw_data1)

print(f"Shape: {df_raw1.shape}")
df_raw1.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,9dd3879e-2b0c-4058-8c8c-19f6b376f69c,"The worst app, also bank am begging for my own...",1,2026-05-16 12:35:22,BOA Bank,Google Play
1,5f69466d-ec06-4eb5-816c-296accffeff2,was Good 🙏,5,2026-05-16 00:10:06,BOA Bank,Google Play
2,f9246b8a-6688-4249-b804-0b0c5dd60590,cool,5,2026-05-15 21:07:21,BOA Bank,Google Play
3,21bcff26-5b05-485d-958f-4832ec1fac01,Its Good,5,2026-05-15 15:01:09,BOA Bank,Google Play
4,c5eb7589-59b7-4d72-8aa9-100a703ecaa3,good,5,2026-05-14 21:18:44,BOA Bank,Google Play


Exploring raw data

In [6]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw1)}")
print(f"\nColumn dtypes:")
print(df_raw1.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [7]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts1 = df_raw1['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts1.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  280  ████████████████████████████████████████████████████████
  4 stars:   37  ███████
  3 stars:   18  ███
  2 stars:   16  ███
  1 stars:  149  █████████████████████████████


In [8]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw1['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw1['date'].dtype}")

Sample date values (raw):
0   2026-05-16 12:35:22
1   2026-05-16 00:10:06
2   2026-05-15 21:07:21
3   2026-05-15 15:01:09
4   2026-05-14 21:18:44
5   2026-05-12 11:50:32
6   2026-05-11 18:18:54
7   2026-05-09 14:41:50
8   2026-05-08 13:47:07
9   2026-05-07 10:33:06

Date dtype: datetime64[us]


DATA QUALITY AUDIT

In [9]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw1.isnull().sum()
missing_pct = (missing / len(df_raw1) * 100).round(2)

for col in df_raw1.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [10]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes1 = df_raw1.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes1}")

# Duplicate review IDs
id_dupes1 = df_raw1.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes1}")

# Empty reviews (also a form of bad data)
empty_reviews1 = (df_raw1['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews1}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 90
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [11]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw1['date'].dtype}")
print(f"  Sample values: {df_raw1['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-05-16 12:35:22
  Target format: YYYY-MM-DD (string or date object)


CLEANING STRATEGY

In [12]:
# Work on a copy so raw data stays untouched
df1 = df_raw1.copy()

print(f"Starting with: {len(df1)} reviews")

Starting with: 500 reviews


In [13]:
before1 = len(df1)

# Drop rows missing the critical columns
critical_cols1 = ['review', 'rating']
df = df1.dropna(subset=critical_cols1)

removed = before1 - len(df1)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df1)} reviews")

Removed 0 rows with missing critical data
Remaining: 500 reviews


remove duplIcates

In [14]:
before1 = len(df1)

df1 = df1.drop_duplicates(subset=['review_id'], keep='first')

removed = before1 - len(df1)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df1)} reviews")

Removed 0 duplicate reviews
Remaining: 500 reviews


normalizing dates

In [15]:
print("Before normalization:")
print(df1['date'].head(3).to_string())
print(f"dtype: {df1['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df1['date'] = pd.to_datetime(df1['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df1['date'].head(3).to_string())
print(f"dtype: {df1['date'].dtype}")

print(f"\nDate range: {df1['date'].min()} to {df1['date'].max()}")

Before normalization:
0   2026-05-16 12:35:22
1   2026-05-16 00:10:06
2   2026-05-15 21:07:21
dtype: datetime64[us]

After normalization:
0    2026-05-16
1    2026-05-16
2    2026-05-15
dtype: str

Date range: 2025-02-23 to 2026-05-16


clean review text

In [16]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


VALIDATE Ratings

In [17]:
# Check for out-of-range ratings
invalid_ratings1 = df1[(df1['rating'] < 1) | (df1['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings1)}")

# Remove them
df1 = df1[(df1['rating'] >= 1) & (df1['rating'] <= 5)]

# Ensure rating is stored as integer
df1['rating'] = df1['rating'].astype(int)

print(f"Remaining: {len(df1)} reviews")
print(f"Rating dtype: {df1['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64


FINAL OUTPUT

In [18]:
# Select only the 5 required columns in the right order
df_clean1 = df1[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean1 = df_clean1.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean1.shape}")
df_clean1.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,"The worst app, also bank am begging for my own...",1,2026-05-16,BOA Bank,Google Play
1,was Good 🙏,5,2026-05-16,BOA Bank,Google Play
2,cool,5,2026-05-15,BOA Bank,Google Play
3,Its Good,5,2026-05-15,BOA Bank,Google Play
4,good,5,2026-05-14,BOA Bank,Google Play
5,it's very good app,5,2026-05-12,BOA Bank,Google Play
6,this app is good but the speed of app is very ...,2,2026-05-11,BOA Bank,Google Play
7,good,5,2026-05-09,BOA Bank,Google Play
8,boa the best,5,2026-05-08,BOA Bank,Google Play
9,bank of absiniya is best bank in ethiopian,5,2026-05-07,BOA Bank,Google Play


In [19]:
# Save to CSV
import os
os.makedirs('Data/processed', exist_ok=True)

output_path1 = 'Data/processed/BOA_bank_reviews_clean.csv'
df_clean1.to_csv(output_path1, index=False)

print(f"Saved to: {output_path1}")

Saved to: Data/processed/BOA_bank_reviews_clean.csv


Preprocessing report for BOA

In [20]:
print("=" * 55)
print("  PREPROCESSING REPORT — BOA Bank Reviews")
print("=" * 55)

original_count1 = len(df_raw1)
final_count1    = len(df_clean1)
removed_total1 = original_count1 - final_count1
retention_rate1 = (final_count1 / original_count1 * 100)

print(f"\n  Raw reviews collected  : {original_count1:>6}")
print(f"  Reviews after cleaning : {final_count1:>6}")
print(f"  Reviews removed        : {removed_total1:>6}")
print(f"  Data retention rate    : {retention_rate1:>5.1f}%")

quality1 = "EXCELLENT" if retention_rate1 >= 95 else ("GOOD" if retention_rate1 >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality1}")

print(f"\n  Date range : {df_clean1['date'].min()}  to  {df_clean1['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_clean1['rating'].unique(), reverse=True):
    count = (df_clean1['rating'] == rating).sum()
    pct   = count / final_count1 * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_clean1['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_clean1.columns:
    print(f"    - {col}")

print("\n" + "=" * 55)

  PREPROCESSING REPORT — BOA Bank Reviews

  Raw reviews collected  :    500
  Reviews after cleaning :    500
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-02-23  to  2026-05-16

  Rating distribution:
    5 stars :  280 (56.0%)  ████████████████████████████████████████████████████████
    4 stars :   37 ( 7.4%)  ███████
    3 stars :   18 ( 3.6%)  ███
    2 stars :   16 ( 3.2%)  ███
    1 stars :  149 (29.8%)  █████████████████████████████

  Text length stats:
    Min    : 1 characters
    Median : 15 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source



Awash

In [21]:
# The unique identifier for Awash Bank's app on the Google Play Store
AWASH_APP_ID = 'com.sc.awashpay'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    AWASH_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("Awash Bank App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

Awash Bank App Info
App Title   : AwashBIRR Pro
Current Score: 4.339221
Total Ratings: 18,857
Total Reviews: 3,420
Installs     : 1,000,000+


In [22]:
# Step 2: Scrape reviews
print(f"Scraping reviews for Awash Bank...")

result, continuation_token = reviews(
    AWASH_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=500,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for Awash Bank...
Collected 500 raw reviews


In [23]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 65015a21-bb26-4ddb-9ad5-e39629b685d7
  userName: Fitsum Chicha lule
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocKTUIHw6MzNfuV13H0B3OeOFBkJX0gtm34ARqipt_XEFvVgYg=mo
  content: so fast and easy to use
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 1.1.3
  at: 2026-05-17 11:40:49
  replyContent: None
  repliedAt: None
  appVersion: 1.1.3


In [24]:
 # Step 3: Extract only the columns we need
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Awash Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,65015a21-bb26-4ddb-9ad5-e39629b685d7,so fast and easy to use,5,2026-05-17 11:40:49,Awash Bank,Google Play
1,837986c5-f6c8-4422-8e79-b4c516d4c4a9,but kobo town water bill payment is not included,5,2026-05-16 17:06:10,Awash Bank,Google Play
2,45be65f5-4758-4f11-ab71-801b85ec31b9,Bad application,1,2026-05-16 13:54:35,Awash Bank,Google Play
3,c653753f-9d7b-4f1a-a433-cbf99190082c,good app 👍,5,2026-05-15 08:00:26,Awash Bank,Google Play
4,fe879501-be94-4a3b-9e5d-4295b24d04e8,good,5,2026-05-14 18:48:44,Awash Bank,Google Play


In [25]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [26]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  370  ██████████████████████████████████████████████████████████████████████████
  4 stars:   50  ██████████
  3 stars:   12  ██
  2 stars:   12  ██
  1 stars:   56  ███████████


In [27]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-17 11:40:49
1   2026-05-16 17:06:10
2   2026-05-16 13:54:35
3   2026-05-15 08:00:26
4   2026-05-14 18:48:44
5   2026-05-13 14:06:56
6   2026-05-13 09:11:38
7   2026-05-12 15:29:21
8   2026-05-10 12:51:30
9   2026-05-09 16:23:41

Date dtype: datetime64[us]


In [28]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [29]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 97
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [30]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-05-17 11:40:49
  Target format: YYYY-MM-DD (string or date object)


In [31]:
# Work on a copy so raw data stays untouched
df = df_raw.copy()

print(f"Starting with: {len(df)} reviews")

Starting with: 500 reviews


In [32]:
before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

Removed 0 rows with missing critical data
Remaining: 500 reviews


In [33]:
before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

Removed 0 duplicate reviews
Remaining: 500 reviews


In [34]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-05-17 11:40:49
1   2026-05-16 17:06:10
2   2026-05-16 13:54:35
dtype: datetime64[us]

After normalization:
0    2026-05-17
1    2026-05-16
2    2026-05-16
dtype: str

Date range: 2025-05-11 to 2026-05-17


In [35]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


In [36]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64


In [37]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,so fast and easy to use,5,2026-05-17,Awash Bank,Google Play
1,but kobo town water bill payment is not included,5,2026-05-16,Awash Bank,Google Play
2,Bad application,1,2026-05-16,Awash Bank,Google Play
3,good app 👍,5,2026-05-15,Awash Bank,Google Play
4,good,5,2026-05-14,Awash Bank,Google Play
5,This app is the best to use. thank you! commen...,5,2026-05-13,Awash Bank,Google Play
6,tajaajila torban tokko osoo hinkenniin network...,2,2026-05-13,Awash Bank,Google Play
7,it's good,5,2026-05-12,Awash Bank,Google Play
8,happy,5,2026-05-10,Awash Bank,Google Play
9,nice,5,2026-05-09,Awash Bank,Google Play


In [38]:
# Save to CSV
import os
os.makedirs('Data/processed', exist_ok=True)

output_path = 'Data/processed/awash_bank_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: Data/processed/awash_bank_reviews_clean.csv


In [39]:
print("=" * 55)
print("  PREPROCESSING REPORT — Awash Bank Reviews")
print("=" * 55)

original_count = len(df_raw)
final_count    = len(df_clean)
removed_total  = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\n  Raw reviews collected  : {original_count:>6}")
print(f"  Reviews after cleaning : {final_count:>6}")
print(f"  Reviews removed        : {removed_total:>6}")
print(f"  Data retention rate    : {retention_rate:>5.1f}%")

quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality}")

print(f"\n  Date range : {df_clean['date'].min()}  to  {df_clean['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_clean['rating'].unique(), reverse=True):
    count = (df_clean['rating'] == rating).sum()
    pct   = count / final_count * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_clean['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_clean.columns:
    print(f"    - {col}")

print("\n" + "=" * 55)

  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :    500
  Reviews after cleaning :    500
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-05-11  to  2026-05-17

  Rating distribution:
    5 stars :  370 (74.0%)  ██████████████████████████████████████████████████████████████████████████
    4 stars :   50 (10.0%)  ██████████
    3 stars :   12 ( 2.4%)  ██
    2 stars :   12 ( 2.4%)  ██
    1 stars :   56 (11.2%)  ███████████

  Text length stats:
    Min    : 1 characters
    Median : 14 characters
    Max    : 492 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source



CBE

In [40]:
# The unique identifier for CBE Bank's app on the Google Play Store
CBE_APP_ID = 'com.combanketh.mobilebanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info3 = app(
    CBE_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("CBE Bank App Info")
print("=" * 50)
print(f"App Title   : {app_info3['title']}")
print(f"Current Score: {app_info3['score']}")
print(f"Total Ratings: {app_info3['ratings']:,}")
print(f"Total Reviews: {app_info3['reviews']:,}")
print(f"Installs     : {app_info3['installs']}")

CBE Bank App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.284731
Total Ratings: 48,534
Total Reviews: 9,327
Installs     : 10,000,000+


In [42]:
# Step 2: Scrape reviews
print(f"Scraping reviews for CBE Bank...")

result2, continuation_token = reviews(
    CBE_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=500,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for CBE Bank...
Collected 500 raw reviews


In [43]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result2[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result2[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 7b8203d4-e95a-4693-b2ea-3e35a6aa797d
  userName: Cadnaan Axmed
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocK-Qvs_djMcMGYEbXQgdQlSpSGFazBDrXl9OnncnQS5IvtGGw=mo
  content: f*k
  score: 1
  thumbsUpCount: 0
  reviewCreatedVersion: 5.3.0
  at: 2026-05-17 13:23:30
  replyContent: None
  repliedAt: None
  appVersion: 5.3.0


In [45]:
# Step 3: Extract only the columns we need
raw_data2 = []

for r in result2:
    raw_data2.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'CBE Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw2 = pd.DataFrame(raw_data2)

print(f"Shape: {df_raw2.shape}")
df_raw2.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,7b8203d4-e95a-4693-b2ea-3e35a6aa797d,f*k,1,2026-05-17 13:23:30,CBE Bank,Google Play
1,60e3a921-2d6b-44a8-bb4b-636af4637b83,The Bank You can Always Rely on!,5,2026-05-17 13:10:51,CBE Bank,Google Play
2,92e988a8-485c-4691-90ff-4005e0f0b3b7,Please make the CBE Noor toggle to be optional...,2,2026-05-17 09:25:49,CBE Bank,Google Play
3,d7c8285b-d811-4738-ba6e-98014135b5a3,Amazing App,5,2026-05-17 04:11:07,CBE Bank,Google Play
4,850f1e2d-aae6-421d-a6b3-c9432282494e,It stopped working on its own. When you check ...,1,2026-05-16 23:44:24,CBE Bank,Google Play


In [46]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw2)}")
print(f"\nColumn dtypes:")
print(df_raw2.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [47]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts2 = df_raw2['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts2.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  334  ██████████████████████████████████████████████████████████████████
  4 stars:   45  █████████
  3 stars:   33  ██████
  2 stars:   13  ██
  1 stars:   75  ███████████████


In [48]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw2['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw2['date'].dtype}")

Sample date values (raw):
0   2026-05-17 13:23:30
1   2026-05-17 13:10:51
2   2026-05-17 09:25:49
3   2026-05-17 04:11:07
4   2026-05-16 23:44:24
5   2026-05-16 22:47:39
6   2026-05-16 21:47:39
7   2026-05-16 19:03:11
8   2026-05-16 15:50:50
9   2026-05-16 12:15:55

Date dtype: datetime64[us]


In [49]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing2 = df_raw2.isnull().sum()
missing_pct2 = (missing2 / len(df_raw2) * 100).round(2)

for col in df_raw2.columns:
    status = f"{missing2[col]} missing ({missing_pct2[col]}%)" if missing2[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [50]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes2 = df_raw2.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes2}")

# Duplicate review IDs
id_dupes2 = df_raw2.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes2}")

# Empty reviews (also a form of bad data)
empty_reviews2 = (df_raw2['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews2}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 124
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [51]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw2['date'].dtype}")
print(f"  Sample values: {df_raw2['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-05-17 13:23:30
  Target format: YYYY-MM-DD (string or date object)


In [52]:
# Work on a copy so raw data stays untouched
df2 = df_raw2.copy()

print(f"Starting with: {len(df2)} reviews")

Starting with: 500 reviews


In [53]:
before2 = len(df2)

# Drop rows missing the critical columns
critical_cols2 = ['review', 'rating']
df2 = df2.dropna(subset=critical_cols2)

removed2 = before2 - len(df2)
print(f"Removed {removed2} rows with missing critical data")
print(f"Remaining: {len(df2)} reviews")

Removed 0 rows with missing critical data
Remaining: 500 reviews


In [54]:
print("Before normalization:")
print(df2['date'].head(3).to_string())
print(f"dtype: {df2['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df2['date'] = pd.to_datetime(df2['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df2['date'].head(3).to_string())
print(f"dtype: {df2['date'].dtype}")

print(f"\nDate range: {df2['date'].min()} to {df2['date'].max()}")

Before normalization:
0   2026-05-17 13:23:30
1   2026-05-17 13:10:51
2   2026-05-17 09:25:49
dtype: datetime64[us]

After normalization:
0    2026-05-17
1    2026-05-17
2    2026-05-17
dtype: str

Date range: 2026-03-05 to 2026-05-17


In [55]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw2 = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw2)}")
print(f"After : {repr(clean_text(sample_raw2))}")

# Apply to the full column
df2['review'] = df2['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before2 = len(df2)
df2 = df2[df['review'].str.len() > 0]
removed2 = before2 - len(df2)
print(f"\nRemoved {removed2} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


In [56]:
# Check for out-of-range ratings
invalid_ratings2 = df2[(df2['rating'] < 1) | (df2['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings2)}")

# Remove them
df2 = df2[(df2['rating'] >= 1) & (df2['rating'] <= 5)]

# Ensure rating is stored as integer
df2['rating'] = df2['rating'].astype(int)

print(f"Remaining: {len(df2)} reviews")
print(f"Rating dtype: {df2['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64


In [57]:
# Select only the 5 required columns in the right order
df_clean2 = df2[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean2 = df_clean2.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean2.shape}")
df_clean2.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,f*k,1,2026-05-17,CBE Bank,Google Play
1,Amazing App,5,2026-05-17,CBE Bank,Google Play
2,The Bank You can Always Rely on!,5,2026-05-17,CBE Bank,Google Play
3,Please make the CBE Noor toggle to be optional...,2,2026-05-17,CBE Bank,Google Play
4,The most backward and unstable financial app i...,1,2026-05-16,CBE Bank,Google Play
5,ok,5,2026-05-16,CBE Bank,Google Play
6,Good,5,2026-05-16,CBE Bank,Google Play
7,🤙🏼🤙🏼,5,2026-05-16,CBE Bank,Google Play
8,worst,1,2026-05-16,CBE Bank,Google Play
9,this app very full,5,2026-05-16,CBE Bank,Google Play


In [58]:
# Save to CSV
import os
os.makedirs('Data/processed', exist_ok=True)

output_path = 'Data/processed/cbe_bank_reviews_clean.csv'
df_clean2.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: Data/processed/cbe_bank_reviews_clean.csv


In [59]:
print("=" * 55)
print("  PREPROCESSING REPORT — CBE Bank Reviews")
print("=" * 55)

original_count = len(df_raw)
final_count    = len(df_clean)
removed_total  = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\n  Raw reviews collected  : {original_count:>6}")
print(f"  Reviews after cleaning : {final_count:>6}")
print(f"  Reviews removed        : {removed_total:>6}")
print(f"  Data retention rate    : {retention_rate:>5.1f}%")

quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality}")

print(f"\n  Date range : {df_clean['date'].min()}  to  {df_clean['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_clean['rating'].unique(), reverse=True):
    count = (df_clean['rating'] == rating).sum()
    pct   = count / final_count * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_clean['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_clean.columns:
    print(f"    - {col}")

print("\n" + "=" * 55)

  PREPROCESSING REPORT — CBE Bank Reviews

  Raw reviews collected  :    500
  Reviews after cleaning :    500
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-05-11  to  2026-05-17

  Rating distribution:
    5 stars :  370 (74.0%)  ██████████████████████████████████████████████████████████████████████████
    4 stars :   50 (10.0%)  ██████████
    3 stars :   12 ( 2.4%)  ██
    2 stars :   12 ( 2.4%)  ██
    1 stars :   56 (11.2%)  ███████████

  Text length stats:
    Min    : 1 characters
    Median : 14 characters
    Max    : 492 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

